# Diebold-Mariano tests and HAC serial correlation

A lower average loss is not enough to claim forecast superiority. This notebook uses the production pairwise test in `covharness.inference`. It does not implement SPA, MCS, or Giacomini-White.

The loss differential is $d_t = L_{A,t} - L_{B,t}$. Negative values are dates on which A wins. The null is $\mathrm{E}[d_t]=0$. Serial correlation in $d_t$ requires a HAC standard error. Long streaks of wins are not independent evidence.

The current baseline remains Bartlett / Newey-West HAC with $L=\lfloor 4(T/100)^{2/9}\rfloor$ and a $N(0,1)$ reference. Harvey-Leybourne-Newbold is not applied. The expanded synthetic calibration does not change that default. Load stored grid results rather than re-running $B=5000$ here.


In [ ]:
from covharness.inference import simulate_hac_size_power, plot_hac_size_power

result = simulate_hac_size_power()
result

Under an IID null both the naive $t$-test and HAC DM reject near 5 percent. Under a mean-zero AR(1) null the naive test over-rejects. HAC DM is closer to nominal size, but the recorded $\rho=0.6$, $T=250$ cell remains oversized. A negative mean shift is rejected in the A-better direction. Harvey-Leybourne-Newbold is not applied. Clark-West is a separate nested scalar squared-error procedure and is not used here.

These Monte Carlo rejection rates describe the finite-sample size of the implemented procedure. The Monte Carlo interval describes simulation uncertainty. Neither object is an empirical forecast-comparison $p$-value.


In [ ]:
plot_hac_size_power(result, "../results/dm_hac_size.png")

The expanded null grid is pre-specified. $\rho\in\{0,0.3,0.6,0.8,0.9\}$, $T\in\{250,500,1000\}$, and lag rules $L=0$, $L_{\mathrm{auto}}$, $2L_{\mathrm{auto}}$, $4L_{\mathrm{auto}}$. $B=5000$ and seed $20260916$. Paths are reused across lag rules within each $(T,\rho)$ cell. The table below is stored output. It is not a bandwidth search.

In [ ]:
from pathlib import Path

import pandas as pd

table_path = Path("../results/dm_hac_calibration_sensitivity.csv")
table = pd.read_csv(table_path)
auto = table.loc[table["lag_rule"] == "L_auto", ["T", "rho", "hac_lag", "rejection_rate", "mcse", "ci95_lower", "ci95_upper", "mean_estimated_lrv_ratio", "median_estimated_lrv_ratio"]]
auto

The current automatic lag is close to nominal under $\rho=0$. It becomes moderately oversized at moderate persistence and severely oversized at $\rho=0.8$ and $\rho=0.9$. Mean estimated long-run variance relative to $1/(1-\rho)^2$ falls as persistence rises. That is a diagnostic of underestimation, not a license to change the implemented statistic here.

Descriptive figures are stored at `../results/dm_hac_calibration_size_vs_rho.png` and `../results/dm_hac_calibration_bandwidth.png`. No lag rule is labeled as preferred.

In [ ]:
from IPython.display import Image

Image("../results/dm_hac_calibration_size_vs_rho.png")

In [ ]:
Image("../results/dm_hac_calibration_bandwidth.png")